# Day 72: Pandas Basics -- Series & DataFrame

Pandas is the cornerstone library for structured data analysis in Python. Its two core data structures -- **Series** (1-D) and **DataFrame** (2-D) -- are analogous to:

| Pandas | C++ STL Equivalent | Description |
|--------|-------------------|-------------|
| `Series` | `std::map<K,V>` or `std::vector` + index | Labeled 1-D array |
| `DataFrame` | `std::map<string, vector>` / custom table class | Labeled 2-D heterogeneous table |
| `Index` | `std::vector<K>` (row/column labels) | Immutable label array |

In C++ you might hand-roll a table class with `std::vector<std::vector<double>>` and separate column-name logic; pandas gives you this out of the box with optimized C/NumPy internals.

## 1. Creating a Series

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# From a list with explicit index labels
ser1 = pd.Series(data=[120, 380, 250, 360],
                 index=['Q1', 'Q2', 'Q3', 'Q4'])
ser1

In [ ]:
# From a dict -- keys become the index
ser2 = pd.Series({'Q1': 320, 'Q2': 180, 'Q3': 300, 'Q4': 405})
ser2

> **C++ comparison:** In C++ you might write `std::map<std::string, int> sales = {{\"Q1\", 320}, ...};` -- pandas `Series` adds vectorized math, alignment by label, and plotting on top.

## 2. Series Operations

### 2.1 Scalar operations

In [ ]:
ser1 += 10
ser1

### 2.2 Vector (element-wise) operations -- automatic alignment by label

In [ ]:
# ser1 and ser2 share the same index labels, so pandas aligns automatically
ser1 + ser2

> **Enterprise example:** Imagine two quarterly revenue reports with slightly different quarters -- pandas aligns on label, filling `NaN` where data is missing, unlike a raw C++ loop that would need explicit index matching.

### 2.3 Indexing and slicing

In [ ]:
# Integer positional index
print("Position 2:", ser1.iloc[2])

# Label-based index
print("Q3 label:", ser1['Q3'])

In [ ]:
# Slicing with labels (end-inclusive! unlike Python lists)
ser2['Q2':'Q4']

In [ ]:
# Fancy (label list) indexing
ser2[['Q2', 'Q4']]

In [ ]:
# Boolean filtering
ser2[ser2 >= 300]

## 3. Series Attributes & Statistics

In [ ]:
print(f"dtype:       {ser2.dtype}")
print(f"has NaN:     {ser2.hasnans}")
print(f"index:       {ser2.index.tolist()}")
print(f"values:      {ser2.values}")
print(f"is unique:   {ser2.is_unique}")
print(f"size:        {ser2.size}")

In [ ]:
# Descriptive statistics -- one-stop shop
ser2.describe()

In [ ]:
# Individual stats
print(f"sum:    {ser2.sum()}")
print(f"mean:   {ser2.mean()}")
print(f"median: {ser2.median()}")
print(f"std:    {ser2.std():.2f}")
print(f"var:    {ser2.var():.2f}")

### Value counts and mode

In [ ]:
ser3 = pd.Series(['apple', 'banana', 'apple', 'pitaya', 'apple', 'pitaya', 'durian'])
print("Value counts:")
print(ser3.value_counts())
print(f"\nUnique count: {ser3.nunique()}")
print(f"Mode: {ser3.mode().iloc[0]}")

## 4. Handling Missing Data

In [ ]:
ser4 = pd.Series([10, 20, np.nan, 30, np.nan])
print("isna():")
print(ser4.isna())
print("\nnotna():")
print(ser4.notna())

In [ ]:
# Drop NaN
print("dropna():", ser4.dropna().tolist())

# Fill with constant
print("fillna(40):", ser4.fillna(40).tolist())

# Forward-fill
print("ffill:    ", ser4.fillna(method='ffill').tolist())

> **Enterprise example:** Sensor data often has gaps. `fillna(method='ffill')` propagates the last valid reading forward -- a common pattern in IoT time-series pipelines.

## 5. Data Transformation: `map()`, `apply()`, `where()`, `mask()`

In [ ]:
# map() with a dictionary -- like a lookup table
ser6 = pd.Series(['cat', 'dog', np.nan, 'rabbit'])
ser6.map({'cat': 'kitten', 'dog': 'puppy'})

In [ ]:
# apply() with a function -- like std::transform
ser7 = pd.Series([20, 21, 12], index=['London', 'New York', 'Helsinki'])
ser7.apply(lambda x: x ** 2)

In [ ]:
# where() -- keep values that satisfy condition, replace others
ser5 = pd.Series(range(5))
print("where(>1, 10):", ser5.where(ser5 > 1, 10).tolist())

# mask() -- replace values that satisfy condition
print("mask(>1, 10): ", ser5.mask(ser5 > 1, 10).tolist())

> **C++ comparison:** `apply()` is analogous to `std::transform` or `std::for_each` on a vector; `map()` with a dict is like a `std::unordered_map` lookup applied element-wise.

## 6. Sorting & Top-N

In [ ]:
ser8 = pd.Series(
    data=[35, 96, 12, 57, 25, 89],
    index=['grape', 'banana', 'pitaya', 'apple', 'peach', 'orange']
)
print("Sort by value (ascending):")
print(ser8.sort_values())
print("\nSort by index (descending):")
print(ser8.sort_index(ascending=False))

In [ ]:
print("Top 3 largest:")
print(ser8.nlargest(3))
print("\nBottom 2 smallest:")
print(ser8.nsmallest(2))

> **C++ comparison:** `nlargest`/`nsmallest` use a partial sort (heap-based) -- more efficient than a full `std::sort` when you only need top-N.

## 7. Duplicates

In [ ]:
print("Duplicated flags:")
print(ser3.duplicated())
print("\nAfter drop_duplicates():")
print(ser3.drop_duplicates())

## 8. Plotting with Series

In [ ]:
import matplotlib.pyplot as plt

ser9 = pd.Series({'Q1': 400, 'Q2': 520, 'Q3': 180, 'Q4': 380})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
ser9.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_ylim(0, 600)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for i, v in enumerate(ser9):
    axes[0].text(i, v + 5, str(v), ha='center')
axes[0].set_title('Quarterly Sales - Bar')

# Pie chart
ser9.plot(kind='pie', ax=axes[1], autopct='%.1f%%', pctdistance=0.65)
axes[1].set_ylabel('')
axes[1].set_title('Quarterly Sales - Pie')

plt.tight_layout()
plt.show()

---

## 9. DataFrame Basics

A `DataFrame` is a 2-D labeled table with **heterogeneous** column types (unlike a NumPy 2-D array which requires a single dtype). Think of it as a `std::map<string, std::vector<Variant>>` in C++, where each column can hold a different type.

### 9.1 Creating a DataFrame

In [ ]:
# From a dict of lists -- most common pattern
scores = {
    'Math':    [95, 65, 86, 66, 87],
    'English': [66, 75, 82, 69, 82],
    'Chinese': [62, 72, 93, 88, 93],
}
df1 = pd.DataFrame(data=scores, index=np.arange(1001, 1006))
df1

In [ ]:
# From a 2-D NumPy array with explicit column/index labels
arr = np.random.randint(60, 101, (5, 3))
df2 = pd.DataFrame(data=arr, columns=['Math', 'English', 'Chinese'], index=np.arange(1001, 1006))
df2

### 9.2 Key DataFrame Attributes

In [ ]:
print(f"Shape:  {df1.shape}")
print(f"Dtypes:\n{df1.dtypes}")
print(f"\nIndex:  {df1.index.tolist()}")
print(f"Columns: {df1.columns.tolist()}")
print(f"Size:   {df1.size}")

### 9.3 Selecting Columns

In [ ]:
# Single column -> returns a Series
print(type(df1['Math']))
df1['Math']

In [ ]:
# Multiple columns -> returns a DataFrame
df1[['Math', 'English']]

### 9.4 Row Selection: `loc` (label) vs `iloc` (position)

In [ ]:
# loc -- by label
df1.loc[1001]

In [ ]:
# iloc -- by integer position
df1.iloc[0:3]

### 9.5 Filtering Rows (Boolean Indexing)

In [ ]:
# Students with Math score >= 80
df1[df1['Math'] >= 80]

In [ ]:
# Compound condition
df1[(df1['Math'] >= 80) & (df1['English'] >= 80)]

### 9.6 Adding / Modifying Columns

In [ ]:
# Add a computed column
df1['Total'] = df1.sum(axis=1)
df1['Average'] = df1[['Math', 'English', 'Chinese']].mean(axis=1).round(1)
df1

> **Enterprise example:** In a sales pipeline, you might compute `Revenue - Cost` to derive `Profit` as a new column -- pandas vectorizes this across all rows at C speed.

### 9.7 Descriptive Statistics on a DataFrame

In [ ]:
df1.describe()

In [ ]:
# Column-wise aggregation
print("Column means:")
print(df1[['Math', 'English', 'Chinese']].mean())
print(f"\nOverall average score: {df1[['Math', 'English', 'Chinese']].values.mean():.1f}")

### 9.8 Sorting a DataFrame

In [ ]:
# Sort by Total descending
df1.sort_values(by='Total', ascending=False)

### 9.9 Handling Missing Data in DataFrames

In [ ]:
# Introduce some NaN values
df3 = df1[['Math', 'English', 'Chinese']].copy()
df3.iloc[1, 1] = np.nan   # Student 1002, English
df3.iloc[3, 0] = np.nan   # Student 1004, Math
df3

In [ ]:
# Detect NaN
df3.isna()

In [ ]:
# Fill NaN with column mean -- common enterprise pattern for imputation
df3.fillna(df3.mean())

In [ ]:
# Or drop rows that contain any NaN
df3.dropna()

## 10. Enterprise Example: Employee Database Analysis

Suppose we receive employee data from a database query. This mirrors a typical enterprise ETL scenario.

In [ ]:
# Simulate a database query result
employees = pd.DataFrame({
    'ename': ['Zhang San', 'Li Si', 'Wang Wu', 'Zhao Liu', 'Chen Qi'],
    'job':   ['Engineer', 'Manager', 'Engineer', 'Sales', 'Manager'],
    'sal':   [8000, 12000, 7500, 6000, 13000],
    'comm':  [np.nan, 2000, np.nan, 1500, 3000],
    'dno':   [20, 10, 20, 30, 10],
})
employees

In [ ]:
# Average salary by department
employees.groupby('dno')['sal'].mean()

In [ ]:
# Fill commission NaN with 0 for total-comp calculation
employees['comm'] = employees['comm'].fillna(0)
employees['total_comp'] = employees['sal'] + employees['comm']
employees

In [ ]:
# Top earner per job title
employees.sort_values('total_comp', ascending=False).groupby('job').first()

## 11. Pandas vs C++: Summary

| Feature | Pandas | C++ Equivalent |
|---------|--------|----------------|
| Labeled vector | `Series` | `std::map<string, T>` + manual math |
| 2-D table | `DataFrame` | `std::vector<std::vector<Variant>>` + custom class |
| Boolean filtering | `df[df['col'] > val]` | Manual loop with `if` + `push_back` |
| Aggregation | `df.groupby('col').mean()` | Manual group-by with `std::unordered_map` |
| Missing data | `np.nan` + `fillna()` / `dropna()` | `std::optional<T>` + manual checks |
| Sort | `sort_values()` / `sort_index()` | `std::sort` with custom comparator |
| Vectorized math | Automatic via NumPy | Manual loops or Eigen/Blaze |
| I/O | `read_csv()` / `read_sql()` | Manual parsing or third-party libs |

Pandas achieves C-like performance through its NumPy/Cython backend while providing a high-level, expressive API -- the best of both worlds for data analysis.

---

## Quick Reference: Common Series Methods

| Category | Methods |
|----------|---------|
| **Stats** | `describe()`, `mean()`, `median()`, `std()`, `var()`, `min()`, `max()`, `sum()`, `count()` |
| **Missing** | `isna()`, `notna()`, `dropna()`, `fillna()` |
| **Transform** | `map()`, `apply()`, `where()`, `mask()` |
| **Sorting** | `sort_values()`, `sort_index()`, `nlargest()`, `nsmallest()` |
| **Duplicates** | `duplicated()`, `drop_duplicates()`, `unique()`, `nunique()`, `value_counts()` |
| **Plotting** | `plot(kind='bar'|'pie'|'line')` |